<a href="https://colab.research.google.com/github/IT22283344/DL_Assignment/blob/Hirusha/Garabage_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# SE4050 Deep Learning Assignment - MobileNetV2 Implementation
# Team Member: Sumanaweera A D H R
# Model: MobileNetV2 for Waste Classification
# Dataset: Waste Classification Data (Organic vs Recyclable)
# =============================================================================

# Import  libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os
from datetime import datetime

In [7]:
!apt-get install p7zip-full


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:

import os

seven_z_path = '/content/drive/MyDrive/DATASET.7z'
extract_dir = '/content/DATASET'

# Create extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Extract using 7z command
!7z x "{seven_z_path}" -o"{extract_dir}" -y
print(f"7z file extracted to: {extract_dir}")

# =============================================================================
# DATA PREPARATION AND LOADING
# =============================================================================
train_dir = os.path.join(extract_dir, 'TRAIN')
test_dir = os.path.join(extract_dir, 'TEST')

print("Training directory:", train_dir)
print("Test directory:", test_dir)

# Function to count images in 'organic(O)' and 'recyclable(R)' folders
def count_images(directory):
    organic_count = len(os.listdir(os.path.join(directory, 'O')))
    recyclable_count = len(os.listdir(os.path.join(directory, 'R')))
    return organic_count, recyclable_count

# Count and print the number of images
try:
    train_organic, train_recyclable = count_images(train_dir)
    test_organic, test_recyclable = count_images(test_dir)

    print(f"\nTraining Data: {train_organic} organic, {train_recyclable} recyclable")
    print(f"Test Data: {test_organic} organic, {test_recyclable} recyclable")
    print(f"Total Images: {train_organic + train_recyclable + test_organic + test_recyclable}")

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please check your dataset paths and folder structure!")



7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/                                 1 file, 216000968 bytes (206 MiB)

Extracting archive: /content/drive/MyDrive/DATASET.7z
--
Path = /content/drive/MyDrive/DATASET.7z
Type = 7z
Physical Size = 216000968
Headers Size = 305148
Method = LZMA:23
Solid = +
Blocks = 1

  0%      1%      1% 249 - DATASET/TEST/O/O_12816.jpg                                       2% 421 - DATASET/TEST/O/O_12988.jpg                                       3% 496 - DATASET/TEST/O/O_13063.jpg     

In [16]:
# =============================================================================
#  DATA PREPROCESSING AND AUGMENTATION
# =============================================================================

# Image parameters for MobileNetV2
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
NUM_CLASSES = 2

# Data Augmentation for Training
# Why augmentation? Prevents overfitting and improves model generalization
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalize pixel values to [0,1]
    rotation_range=30,           # Random rotation up to 30 degrees
    width_shift_range=0.2,       # Random horizontal shift
    height_shift_range=0.2,      # Random vertical shift
    horizontal_flip=True,        # Random horizontal flip
    zoom_range=0.2,              # Random zoom
    shear_range=0.2,             # Random shear
    brightness_range=[0.8, 1.2], # Random brightness adjustment
    fill_mode='nearest'          # Fill missing pixels with nearest
)

# Only rescaling for validation/test data (no augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

In [24]:
# Create data generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # For multi-class classification
    shuffle=True,              # Shuffle training data
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,             # Don't shuffle test data for consistent evaluation
    seed=42
)

# Get class indices and labels
class_names = list(train_generator.class_indices.keys())
print(f" Class names: {class_names}")
print(f" Class indices: {train_generator.class_indices}")

Found 22564 images belonging to 2 classes.
Found 2514 images belonging to 2 classes.
 Class names: ['O', 'R']
 Class indices: {'O': 0, 'R': 1}
